In [1]:
# =============================================================
# PASTE THIS INTO A SINGLE DEEPNOTE CODE CELL AND RUN IT ONCE.
# It fixes the "cannot import name 'genai' from 'google'" error
# and prepares everything needed to launch the Streamlit app.
# =============================================================

# --- Step 1: Force-clean and reinstall the Google GenAI SDKs ---
# Deepnote's base image sometimes ships a partial "google"
# namespace package (from google-cloud-* libs) that shadows
# Google GenAI packages. A normal `pip install` doesn't always fix this,
# so we uninstall anything conflicting and force a clean install.
!

In [7]:
# =============================================================
# RUN THIS CELL AFTER RESTARTING THE KERNEL.
# It verifies the import works, validates the API key if present,
# then writes app.py to your Deepnote project folder.
# =============================================================

import os
from pathlib import Path

from google import genai
from google.genai import types

print("✅ google-genai imports correctly now!")

# IMPORTANT:
# Do NOT overwrite GEMINI_API_KEY with a placeholder like:
# os.environ["GEMINI_API_KEY"] = "enter your api key here"
#
# Instead, set a real key in Deepnote Project settings -> Environment variables
# with the name GEMINI_API_KEY. For quick testing only, uncomment this line
# and replace it with your real key:
# os.environ["GEMINI_API_KEY"] = "AIza...your_real_key_here..."

API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()
PLACEHOLDER_VALUES = {
    "",
    "enter your api key here",
    "your api key here",
    "YOUR_API_KEY",
    "your_api_key",
    "AIza...your_real_key_here...",
}

# --- Quick sanity check that the key + model work, only when a real key exists ---
if not API_KEY or API_KEY in PLACEHOLDER_VALUES:
    print(
        "⚠️ GEMINI_API_KEY is missing or still set to a placeholder.\n"
        "   Skipping the Gemini sanity check, but app.py will still be written.\n"
        "   Set a real GEMINI_API_KEY in Deepnote Project settings -> Environment variables,\n"
        "   or uncomment the os.environ line above and paste a real key."
    )
else:
    try:
        client = genai.Client(api_key=API_KEY)
        test_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=types.GenerateContentConfig(
                system_instruction="You are a helpful assistant. Reply in one short sentence.",
                temperature=0.4,
            ),
        )
        test_response = test_chat.send_message("Say hello in one sentence.")
        print("Model test response:", test_response.text)
    except Exception as e:
        print(
            "⚠️ Gemini sanity check failed, but app.py will still be written.\n"
            f"   Error: {e}\n"
            "   Please verify that GEMINI_API_KEY is a valid key from https://aistudio.google.com/."
        )

# --- Write the full Streamlit app to disk ----------------
# NOTE: The Streamlit app is assembled from a list of normal Python strings.
# This avoids nested triple-quote parsing issues that can cause
# _IncompleteInputError in notebooks when pasting long code cells.
app_lines = [
    'import os',
    'import streamlit as st',
    'from google import genai',
    'from google.genai import types',
    '',
    '# -------------------------',
    '# PAGE SETTINGS',
    '# -------------------------',
    'st.set_page_config(',
    '    page_title="TESSA - IRD Grenada",',
    '    page_icon="🇬🇩",',
    '    layout="wide",',
    ')',
    '',
    '# -------------------------',
    '# API KEY',
    '# -------------------------',
    'def get_api_key():',
    '    """Read Gemini API key from Streamlit secrets first, then environment."""',
    '    try:',
    '        key = st.secrets.get("GEMINI_API_KEY", "")',
    '    except Exception:',
    '        key = ""',
    '',
    '    if not key:',
    '        key = os.environ.get("GEMINI_API_KEY", "")',
    '',
    '    return str(key).strip()',
    '',
    'API_KEY = get_api_key()',
    'PLACEHOLDER_VALUES = {',
    '    "",',
    '    "enter your api key here",',
    '    "your api key here",',
    '    "YOUR_API_KEY",',
    '    "your_api_key",',
    '    "AIza...your_real_key_here...",',
    '}',
    '',
    '# -------------------------',
    '# GEMINI CLIENT / CHAT',
    '# -------------------------',
    'SYSTEM_INSTRUCTION = """',
    'You are TESSA, a professional, friendly virtual assistant for the Inland Revenue Division (IRD) of Grenada.',
    'Your role is to help users understand general tax topics, processes, required documents, deadlines, and where to find official help.',
    '',
    'Important rules:',
    '- Be clear, concise, respectful, and service-oriented.',
    '- If a question requires official legal/tax advice, tell the user to contact IRD Grenada directly.',
    '- Do not invent phone numbers, email addresses, laws, rates, or deadlines.',
    '- If you are unsure, say you are unsure and recommend verifying with the IRD.',
    '- Do not ask for sensitive personal information such as Taxpayer Identification Numbers, passwords, full bank details, or full identity numbers.',
    '- If the user provides sensitive information, tell them not to share it in chat.',
    '"""',
    '',
    '@st.cache_resource(show_spinner=False)',
    'def get_client(api_key: str):',
    '    return genai.Client(api_key=api_key)',
    '',
    '',
    'def new_chat(api_key: str):',
    '    client = get_client(api_key)',
    '    return client.chats.create(',
    '        model="gemini-2.5-flash",',
    '        config=types.GenerateContentConfig(',
    '            system_instruction=SYSTEM_INSTRUCTION,',
    '            temperature=0.3,',
    '        ),',
    '    )',
    '',
    '# -------------------------',
    '# UI',
    '# -------------------------',
    'st.title("🇬🇩 TESSA - IRD Grenada Virtual Assistant")',
    'st.caption("Ask general questions about IRD Grenada services and tax processes. Do not share sensitive personal information.")',
    '',
    'with st.sidebar:',
    '    st.header("About TESSA")',
    '    st.write(',
    '        "TESSA provides general guidance only. For official advice or account-specific support, "',
    '        "please contact the Inland Revenue Division of Grenada directly."',
    '    )',
    '',
    '    if st.button("Start new chat"):',
    '        st.session_state.messages = []',
    '        if API_KEY and API_KEY not in PLACEHOLDER_VALUES:',
    '            st.session_state.chat = new_chat(API_KEY)',
    '        st.rerun()',
    '',
    'if not API_KEY or API_KEY in PLACEHOLDER_VALUES:',
    '    st.error(',
    '        "GEMINI_API_KEY is missing or set to a placeholder. "',
    '        "Set a real key in Deepnote/Streamlit environment variables or .streamlit/secrets.toml."',
    '    )',
    '    st.stop()',
    '',
    'if "messages" not in st.session_state:',
    '    st.session_state.messages = []',
    '',
    'if "chat" not in st.session_state:',
    '    try:',
    '        st.session_state.chat = new_chat(API_KEY)',
    '    except Exception as e:',
    '        st.error(f"Could not initialize Gemini chat: {e}")',
    '        st.stop()',
    '',
    '# Display previous messages',
    'for message in st.session_state.messages:',
    '    with st.chat_message(message["role"]):',
    '        st.markdown(message["content"])',
    '',
    '# Chat input',
    'prompt = st.chat_input("How can TESSA help you today?")',
    '',
    'if prompt:',
    '    st.session_state.messages.append({"role": "user", "content": prompt})',
    '    with st.chat_message("user"):',
    '        st.markdown(prompt)',
    '',
    '    with st.chat_message("assistant"):',
    '        with st.spinner("TESSA is thinking..."):',
    '            try:',
    '                response = st.session_state.chat.send_message(prompt)',
    '                answer = getattr(response, "text", "") or "I am sorry, I could not generate a response."',
    '            except Exception as e:',
    '                answer = f"Sorry, something went wrong while contacting Gemini: {e}"',
    '',
    '        st.markdown(answer)',
    '',
    '    st.session_state.messages.append({"role": "assistant", "content": answer})',
]

app_code = "\n".join(app_lines) + "\n"
Path("app.py").write_text(app_code, encoding="utf-8")

print("✅ app.py has been written successfully.")
print("Run it with: streamlit run app.py")

✅ google-genai imports correctly now!
⚠️ GEMINI_API_KEY is missing or still set to a placeholder.
   Skipping the Gemini sanity check, but app.py will still be written.
   Set a real GEMINI_API_KEY in Deepnote Project settings -> Environment variables,
   or uncomment the os.environ line above and paste a real key.
✅ app.py has been written successfully.
Run it with: streamlit run app.py


In [3]:
# =============================================================
# RUN THIS CELL LAST, to launch the Streamlit app inside Deepnote.
#
# This version:
#   1) Checks whether Streamlit is installed.
#   2) Installs Streamlit automatically if it is missing.
#   3) Launches app.py in the background on port 8501.
#   4) Writes logs to streamlit.log so errors are easy to inspect.
# =============================================================

import os
import sys
import subprocess
import time
from pathlib import Path

# -------------------------
# Confirm app.py exists
# -------------------------
if not Path("app.py").exists():
    raise FileNotFoundError("app.py was not found. Run the cell that writes app.py first.")

# -------------------------
# Ensure Streamlit is installed
# -------------------------
def ensure_package(package_name: str, import_name: str | None = None):
    """Install a package with pip if it cannot be imported."""
    import importlib.util

    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {package_name} because it is not currently available...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", package_name])
        print(f"✅ {package_name} installed successfully.")
    else:
        print(f"✅ {package_name} is already installed.")

ensure_package("streamlit", "streamlit")

# -------------------------
# Launch settings
# -------------------------
PORT = 8501
LOG_PATH = Path("streamlit.log")

# Stop any previous Streamlit process launched by this notebook cell.
if "streamlit_process" in globals() and streamlit_process.poll() is None:
    print(f"Stopping previous Streamlit process with PID {streamlit_process.pid}...")
    streamlit_process.terminate()
    try:
        streamlit_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        streamlit_process.kill()
        streamlit_process.wait(timeout=5)

# Cleanup for any leftover process using the same port.
os.system(f"fuser -k {PORT}/tcp >/dev/null 2>&1 || true")
time.sleep(1)

cmd = [
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.port", str(PORT),
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
]

print("Launching Streamlit with command:")
print(" ".join(cmd))

streamlit_log = open(LOG_PATH, "w", encoding="utf-8")
streamlit_process = subprocess.Popen(
    cmd,
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
    text=True,
    preexec_fn=os.setsid if hasattr(os, "setsid") else None,
)

time.sleep(5)

if streamlit_process.poll() is None:
    print(f"✅ Streamlit is running in the background on port {PORT}.")
    print(f"   PID: {streamlit_process.pid}")
    print("   Open the Deepnote preview / Open app link for port 8501.")
    print(f"   Logs are being written to {LOG_PATH}")
    print("\nTo stop it later, run:")
    print("streamlit_process.terminate()")
else:
    print("❌ Streamlit exited immediately. Last log output:")
    streamlit_log.close()
    if LOG_PATH.exists():
        print(LOG_PATH.read_text(encoding="utf-8"))
    else:
        print("No streamlit.log file was created.")

Installing streamlit because it is not currently available...

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
✅ streamlit installed successfully.
Launching Streamlit with command:
/root/venv/bin/python -m streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true --server.enableCORS false --server.enableXsrfProtection false
✅ Streamlit is running in the background on port 8501.
   PID: 149
   Open the Deepnote preview / Open app link for port 8501.
   Logs are being written to streamlit.log

To stop it later, run:
streamlit_process.terminate()


In [5]:
streamlit_process

<Popen: returncode: None args: ['/root/venv/bin/python', '-m', 'streamlit', ...>

In [7]:
PORT

8501

In [9]:
LOG_PATH

PosixPath('streamlit.log')

In [11]:
streamlit_log

<_io.TextIOWrapper name='streamlit.log' mode='w' encoding='utf-8'>